# Stage 5 G4 Direct-Preservation Auto Run

Run the single cell below in a fresh G4/L4/A100 Colab GPU runtime when you want to continue from the completed scale64 trace-SFT checkpoint without rerunning the long trace-SFT job.

This launches `STAGE5_CURRENT_A100_TARGET=traced_sft_direct_preservation_probe`. That target first performs the loop-1 direct-route precheck internally. If loop-1 already matches base, it skips training. If not, it runs the bounded direct-preservation sweep, then chains confirmation and the lightweight depth-router continuation only after the direct-preservation gate passes.

Drive restore is enabled because the scale64 checkpoint is not guaranteed to exist in a fresh runtime.

In [ ]:
import base64, json, os, time, urllib.request
from google.colab import userdata

gh = userdata.get("GH_TOKEN") or userdata.get("GITHUB_TOKEN")
assert gh, "Missing GH_TOKEN/GITHUB_TOKEN in Colab secrets."

hf = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_HUB_TOKEN")
if hf:
    os.environ["HF_TOKEN"] = hf
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf

os.environ["STAGE5_CURRENT_A100_TARGET"] = "traced_sft_direct_preservation_probe"
os.environ["STAGE5_DIRECT_PRESERVE_DRIVE_BACKUP"] = "1"

url = (
    "https://api.github.com/repos/mshapiro123/recurrent-qwen-svgd/"
    f"contents/colab/CURRENT_A100_BOOTSTRAP_CELL.py?ref=main&t={int(time.time())}"
)
req = urllib.request.Request(
    url,
    headers={
        "Authorization": f"Bearer {gh}",
        "Accept": "application/vnd.github+json",
        "Cache-Control": "no-cache",
    },
)
payload = json.loads(urllib.request.urlopen(req).read().decode("utf-8"))
code = base64.b64decode(payload["content"]).decode("utf-8")

required = [
    "sha_resolved_nested_fetch_v3",
    "traced_sft_direct_preservation_probe",
    "STAGE5_DIRECT_PRESERVE_SWEEP",
    "STAGE5_DIRECT_PRESERVE_CHAIN_CONFIRM",
    "STAGE5_DIRECT_PRESERVE_CHAIN_DEPTH_ROUTER",
    "direct_route_precheck_needs_training",
]
missing = [marker for marker in required if marker not in code]
assert not missing, f"Fetched bootstrap is stale or incomplete: {missing}"

print("Fetched bootstrap sha:", payload.get("sha"))
exec(compile(code, "colab/CURRENT_A100_BOOTSTRAP_CELL.py", "exec"))
